# Karakteristik Ekspresi Gen Berdasarkan Sampel
## Rencana *Preprocessing* dan *Initial Feature Engineering*

**Topik.** Karakteristik ekspresi gen antar sampel, dengan fokus pada reduksi dimensi dan pencarian pola kelompok. Sampel adalah *cell line* kanker; ekspresi gen berupa profil RNA-seq (TPM) dari CCLE; data respons obat GDSC (`LN_IC50`) digunakan sebagai pembanding, bukan tujuan utama.

**Pertanyaan penelitian yang didukung notebook ini**
1. Berapa dimensi yang memadai untuk merepresentasikan variasi ekspresi antar *cell line*?
2. Apakah *cell line* mengelompok menurut profil ekspresinya, dan apakah kelompok tersebut selaras dengan jaringan asal?
3. Apakah kelompok tersebut berbeda dalam sensitivitas obat?
4. Gen apa yang paling berperan membedakan kelompok?

**Prinsip dasar.** Klaster dibentuk hanya dari ekspresi gen. Label jaringan dan respons obat digunakan setelah klaster terbentuk untuk menilai hasilnya.

**Dataset.** `CCLE_RNAseq_rsem_genes_tpm_20180929.txt.gz` (ekspresi gen) dan `GDSC_DATASET.csv` (respons obat dan metadata *cell line*).

## 0. Persiapan dan Pemuatan Data

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

BASE_URL  = "https://raw.githubusercontent.com/RahmadFadhil8/proyek-Kelompok-7-machine-learning-/main/"
PATH_GDSC = BASE_URL + "GDSC_DATASET.csv"
PATH_EXPR = BASE_URL + "CCLE_RNAseq_rsem_genes_tpm_20180929.txt.gz"

gdsc     = pd.read_csv(PATH_GDSC)
expr_raw = pd.read_csv(PATH_EXPR, sep="\t")
print("GDSC :", gdsc.shape)
print("CCLE :", expr_raw.shape)

### 0.1 Profil Data sebagai Dasar Penentuan Rencana

Profil berikut menjadi bukti empiris bagi keputusan pada Bagian 1.

In [ ]:
print("=== CCLE (data inti) ===")
nilai = expr_raw.drop(columns=["gene_id", "transcript_ids"])
v = nilai.to_numpy()
print("Gen x sampel     :", v.shape)
print("Nilai kosong     :", int(np.isnan(v).sum()))
print("Proporsi nilai 0 :", round((v == 0).mean(), 3))
print("Rata-rata TPM    :", round(v.mean(), 2), "| median:", np.median(v), "| maks:", v.max())
aktif = (nilai > 1).mean(axis=1)
print("Gen tidak pernah TPM > 1      :", round((aktif == 0).mean(), 3))
print("Gen aktif di < 10% sampel     :", round((aktif < 0.10).mean(), 3))
cuplikan = pd.Series(rng.choice(v.ravel(), 500_000, replace=False))
print("Skewness TPM mentah:", round(cuplikan.skew(), 1), "| setelah log2(TPM+1):", round(np.log2(cuplikan + 1).skew(), 2))
del v, nilai, cuplikan

print("\n=== GDSC (pembanding) ===")
print("Baris x kolom    :", gdsc.shape)
print("Jumlah cell line :", gdsc["COSMIC_ID"].nunique(), "| nama obat:", gdsc["DRUG_NAME"].nunique(), "| DRUG_ID:", gdsc["DRUG_ID"].nunique())
print("Skewness LN_IC50 :", round(gdsc["LN_IC50"].skew(), 2), "| skewness AUC:", round(gdsc["AUC"].skew(), 2))
kosong_g = (gdsc.isna().mean() * 100).round(2)
display(kosong_g[kosong_g > 0].to_frame("persen_kosong"))

# BAGIAN 1. RENCANA PREPROCESSING

## 1.0 Ringkasan Rencana

Setiap masalah diuraikan menurut tiga pertanyaan: apa masalahnya, mengapa penting, dan bagaimana rencana penanganannya. Penjelasan rinci dan kode terdapat pada subbagian 1.1 sampai 1.7.

**Masalah utama**

| Masalah | Variabel | Rencana Penanganan | Alasan |
|---|---|---|---|
| **Missing value** | Data CCLE: tidak ada. Metadata GDSC: `tissue`, `growth`, `medium` (4,02% sel kosong), `msi` (5,47%), `tcga_label` (21,16%). Matriks `LN_IC50` sel × obat (14,9% kosong) | Metadata diisi kategori `Unknown`; `tcga_label` tidak digunakan. Pada matriks respons, sel dan obat dengan lebih dari 50% nilai kosong dikeluarkan dan sisanya **tidak diimputasi** | Pengisian `Unknown` mempertahankan jumlah sampel. Imputasi nilai respons menghasilkan data sintetis yang dapat mengubah perbandingan antarklaster. `tcga_label` terlalu banyak kosong untuk dijadikan anotasi andal |
| **Outlier** | `TPM` (skewness 174,8; maksimum 225.730); sampel CCLE secara keseluruhan; `LN_IC50` | `log2(TPM+1)` untuk ekspresi. Sampel dengan profil menyimpang hanya **ditandai**, tidak dibuang. `LN_IC50` dinilai dengan aturan IQR **per obat** dan nilai ekstrem dipertahankan | Nilai TPM ekstrem mendominasi jarak antar sampel pada PCA dan klasterisasi. Nilai ekstrem `LN_IC50` mencerminkan sensitivitas biologis yang nyata, dan batas global keliru menandai 96,3% nilai yang sebenarnya wajar bagi obatnya |
| **Data kategorikal** | `tissue_ccle` (26 kategori), `tissue`, `msi`, `growth`, `medium`, `TCGA_DESC` | Kategori dengan proporsi < 1% digabung (`OTHER`/`Other`). Variabel kategorikal digunakan sebagai anotasi dan evaluasi, **bukan** input klasterisasi. Pengkodean *one-hot* hanya diterapkan bila variabel masuk model prediktif | Kategori yang sangat kecil tidak stabil untuk validasi silang. Input klasterisasi berupa komponen numerik sehingga tidak memerlukan pengkodean |
| **Perbedaan skala** | Ekspresi antar gen; `LN_IC50` antar obat | Standardisasi per gen (*z-score*) setelah log; standardisasi `LN_IC50` per obat | Gen dengan varians besar akan mendominasi PCA bila tidak distandardisasi. Median `LN_IC50` antar obat berkisar −5,39 sampai 10,42, jauh melampaui variasi di dalam satu obat |
| **Data leakage** | Seluruh statistik praproses, label jaringan, `LN_IC50`, `AUC`, `Z_SCORE` | Lihat bagian 1.7 | Mencegah evaluasi yang terlalu optimistis dan kesimpulan yang sirkular |

**Penanganan pendukung**

| Masalah | Variabel | Rencana Penanganan | Alasan |
|---|---|---|---|
| Dimensi tinggi dan banyak gen tidak aktif | 57.820 gen; 65,7% aktif di < 10% sampel | Pertahankan gen dengan TPM > 1 pada ≥ 10% sampel | Gen yang hampir tidak pernah aktif tidak memiliki variasi yang berguna dan hanya menambah *noise* |
| Duplikasi pengukuran | Pasangan (`COSMIC_ID`, `DRUG_NAME`) | Rata-rata per pasangan | Satu *cell line* dan satu obat harus menghasilkan satu nilai |
| Kunci antar dataset berbeda | `CELL_LINE_NAME` (GDSC), nama kolom (CCLE) | Pencocokan nama ternormalisasi; ambiguitas diselesaikan dengan kesesuaian jaringan | Mencegah pasangan sel yang keliru |

## 1.1 Penataan Struktur Data

**CCLE.** Data berformat baris = gen dan kolom = sampel, sedangkan analisis memerlukan baris = sampel. Kolom `transcript_ids` tidak diperlukan, dan nama kolom sampel (`NAMA_JARINGAN`, misalnya `A549_LUNG`) memuat label jaringan untuk seluruh 1.019 sampel. Filter gen diterapkan pada skala TPM asli karena ambang `TPM > 1` didefinisikan pada skala tersebut.

**GDSC.** Pemeriksaan duplikasi berdasarkan (`COSMIC_ID`, `DRUG_ID`) menghasilkan nol, tetapi berdasarkan (`COSMIC_ID`, `DRUG_NAME`) terdapat 6.288 duplikat karena sembilan obat memiliki dua `DRUG_ID`. Duplikat diringkas dengan rata-rata (242.035 baris menjadi 235.747 baris). Karena selisih `LN_IC50` antar dua pengukuran cukup besar (median 1,10), uji sensitivitas dengan satu `DRUG_ID` per obat dianjurkan pada tahap lanjut.

In [ ]:
# --- CCLE: struktur, label jaringan, filter gen ---
expr = expr_raw.drop(columns=["transcript_ids"]).set_index("gene_id")
del expr_raw

info_sampel = pd.DataFrame({"kolom_ccle": expr.columns})
info_sampel["nama_sel"]    = info_sampel["kolom_ccle"].str.split("_").str[0]
info_sampel["tissue_ccle"] = info_sampel["kolom_ccle"].str.split("_", n=1).str[1]
info_sampel = info_sampel.set_index("kolom_ccle")
print("Sampel:", len(info_sampel), "| jaringan:", info_sampel["tissue_ccle"].nunique())

AMBANG_TPM, AMBANG_SAMPEL = 1, 0.10
lolos  = (expr > AMBANG_TPM).mean(axis=1) >= AMBANG_SAMPEL
expr_f = expr.loc[lolos]
print(f"Gen: {expr.shape[0]:,} -> {expr_f.shape[0]:,} ({lolos.mean()*100:.1f}% dipertahankan)")
print(f"Proporsi nilai 0: {(expr == 0).to_numpy().mean()*100:.1f}% -> {(expr_f == 0).to_numpy().mean()*100:.1f}%")

# --- GDSC: duplikasi pasangan cell line-obat ---
print("\nDuplikat (COSMIC_ID, DRUG_ID)  :", gdsc.duplicated(["COSMIC_ID", "DRUG_ID"]).sum())
print("Duplikat (COSMIC_ID, DRUG_NAME):", gdsc.duplicated(["COSMIC_ID", "DRUG_NAME"]).sum())
resp = (gdsc.groupby(["COSMIC_ID", "DRUG_NAME"], as_index=False)
            .agg(LN_IC50=("LN_IC50", "mean"), AUC=("AUC", "mean"), Z_SCORE=("Z_SCORE", "mean"),
                 TARGET_PATHWAY=("TARGET_PATHWAY", "first")))
print("Baris GDSC:", gdsc.shape[0], "->", resp.shape[0])

## 1.2 *Missing Value*

**Masalah.** Data CCLE tidak memiliki nilai kosong. Nilai kosong terdapat pada (a) matriks respons *cell line* × obat, karena tidak semua sel diuji dengan semua obat, dan (b) metadata GDSC.

**Mengapa penting.** Nilai kosong yang tidak ditangani membuat ringkasan respons per sel tidak sebanding, dan imputasi yang tidak perlu menambahkan data buatan pada variabel yang hanya berfungsi sebagai pembanding.

**Rencana penanganan.**
- Matriks respons: sel dan obat dengan lebih dari 50% nilai kosong dikeluarkan; sisanya dibiarkan kosong. Ringkasan respons dihitung dengan mengabaikan nilai kosong.
- Metadata: nilai kosong diisi `Unknown`. Kolom `tcga_label` tidak digunakan karena 21,16% sel kosong, sedangkan `TCGA_DESC` dipertahankan sebagai anotasi.
- Kolom `CNA`, `Gene Expression`, dan `Methylation` hanya menandai ketersediaan data, bukan nilai ekspresi, sehingga tidak digunakan.

In [ ]:
# --- matriks respons ---
mat_resp = resp.pivot(index="COSMIC_ID", columns="DRUG_NAME", values="LN_IC50")
print("Matriks awal:", mat_resp.shape, f"| kosong {mat_resp.isna().mean().mean()*100:.1f}%")

BATAS_KOSONG = 0.50
kosong_sel, kosong_obat = mat_resp.isna().mean(axis=1), mat_resp.isna().mean(axis=0)
mat_resp = mat_resp.loc[kosong_sel[kosong_sel <= BATAS_KOSONG].index,
                        kosong_obat[kosong_obat <= BATAS_KOSONG].index]
print("Matriks akhir:", mat_resp.shape, f"| kosong {mat_resp.isna().mean().mean()*100:.1f}%")

# --- metadata GDSC: satu baris per cell line ---
nama_ringkas = {"GDSC Tissue descriptor 1": "tissue", "GDSC Tissue descriptor 2": "tissue2",
                "Microsatellite instability Status (MSI)": "msi", "Growth Properties": "growth",
                "Screen Medium": "medium", "TCGA_DESC": "tcga_desc",
                "Cancer Type (matching TCGA label)": "tcga_label"}
meta = (gdsc.drop_duplicates("COSMIC_ID")[["COSMIC_ID", "CELL_LINE_NAME"] + list(nama_ringkas)]
            .rename(columns=nama_ringkas).set_index("COSMIC_ID"))
display(pd.DataFrame({"jumlah_kosong": meta.isna().sum(), "persen": (meta.isna().mean() * 100).round(2)}))

meta_bersih = meta.drop(columns="tcga_label").copy()
for k in ["tissue", "tissue2", "msi", "growth", "medium", "tcga_desc"]:
    meta_bersih[k] = meta_bersih[k].fillna("Unknown")
print("Nilai kosong tersisa pada metadata:", int(meta_bersih.isna().sum().sum()))

## 1.3 *Outlier*

**Masalah.** Terdapat tiga sumber nilai ekstrem: (a) nilai TPM yang sangat besar (skewness 174,8; maksimum 225.730), (b) sampel yang profil keseluruhannya menyimpang, dan (c) nilai `LN_IC50` yang ekstrem.

**Mengapa penting.** PCA dan klasterisasi peka terhadap jarak sehingga segelintir nilai TPM besar dapat mendominasi hasil. Di sisi lain, penghapusan nilai atau sampel yang secara biologis sahih dapat menghilangkan pola yang justru dicari.

**Rencana penanganan.**
- **TPM:** transformasi `log2(TPM + 1)`. Konstanta 1 mencegah `log(0)`. Skewness turun dari 174,8 menjadi 2,06.
- **Sampel:** tiga pemeriksaan (total TPM, jumlah gen terdeteksi, korelasi terhadap profil median). Sampel di bawah ambang median − 3×MAD hanya **ditandai**, tidak dibuang, karena sampel yang berbeda dapat mencerminkan biologi yang berbeda.
- **`LN_IC50`:** nilai ekstrem dinilai dengan IQR **per obat**, bukan global, karena tiap obat memiliki skala potensi sendiri. Nilai ekstrem tidak dibuang.

In [ ]:
# --- (a) transformasi log ---
expr_log = np.log2(expr_f + 1)
gen_maks = expr_f.max(axis=1).idxmax()
print(f"TPM maks {expr_f.loc[gen_maks].max():,.0f} ({gen_maks}) -> log2(TPM+1) = {expr_log.loc[gen_maks].max():.2f}")

# --- (b) QC sampel (tandai, tidak dibuang) ---
total_tpm       = expr.sum(axis=0)
n_terdeteksi    = (expr > AMBANG_TPM).sum(axis=0)
korelasi_median = expr_log.corrwith(expr_log.median(axis=1))
qc = pd.DataFrame({"total_tpm": total_tpm, "n_gen_terdeteksi": n_terdeteksi,
                   "korelasi_ke_median": korelasi_median}).join(info_sampel["tissue_ccle"])
display(qc[["total_tpm", "n_gen_terdeteksi", "korelasi_ke_median"]].describe().round(3))

mad    = (korelasi_median - korelasi_median.median()).abs().median()
ambang = korelasi_median.median() - 3 * 1.4826 * mad
ditandai = qc[qc["korelasi_ke_median"] < ambang].sort_values("korelasi_ke_median")
print(f"Ambang korelasi: {ambang:.3f} | sampel ditandai: {len(ditandai)}")
display(ditandai.round(3))
del expr, expr_f   # membebaskan memori; analisis selanjutnya memakai expr_log

# --- (c) nilai ekstrem LN_IC50: global vs per obat ---
def batas_iqr(s):
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

lo, hi = batas_iqr(resp["LN_IC50"])
ekstrem_global   = (resp["LN_IC50"] < lo) | (resp["LN_IC50"] > hi)
ekstrem_per_obat = resp.groupby("DRUG_NAME")["LN_IC50"].transform(
    lambda g: (g < batas_iqr(g)[0]) | (g > batas_iqr(g)[1])).astype(bool)
print(f"\nEkstrem global   : {ekstrem_global.sum():>6} ({ekstrem_global.mean()*100:.1f}%)")
print(f"Ekstrem per obat : {ekstrem_per_obat.sum():>6} ({ekstrem_per_obat.mean()*100:.1f}%)")
print(f"Ditandai global tetapi tidak ekstrem pada obatnya: {(ekstrem_global & ~ekstrem_per_obat).sum()} "
      f"({(ekstrem_global & ~ekstrem_per_obat).sum()/ekstrem_global.sum()*100:.1f}% dari yang ditandai global)")

**Temuan:** (1) total TPM per sampel konsisten (999.994,6 sampai 1.000.006,0), sehingga tidak diperlukan normalisasi ulang; (2) korelasi ke profil median berkisar 0,801 sampai 0,942 dan hanya empat sampel berada di bawah ambang 0,816 dengan selisih tipis dan jaringan yang beragam, sehingga tidak ada sampel yang dibuang; (3) dari 10.823 nilai yang ditandai ekstrem secara global, 10.428 (96,3%) tidak ekstrem pada obatnya sendiri, yang menunjukkan bahwa penilaian harus dilakukan per obat.

## 1.4 Data Kategorikal

**Masalah.** Beberapa kategori hanya memiliki sangat sedikit anggota (misalnya `SMALL_INTESTINE` = 1, `SALIVARY_GLAND` = 2). Selain itu, kategori `lung` pada GDSC (7 sel; 0,72%) berisi 6 sel mesotelioma dan 1 sel `Lung_other`, bukan kanker paru sel non-kecil.

**Mengapa penting.** Kategori yang terlalu kecil tidak stabil untuk validasi silang dan evaluasi klaster. Menggabungkan `lung` ke `lung_NSCLC` akan mencampur jenis kanker yang berbeda secara biologis.

**Rencana penanganan.** Kategori dengan proporsi < 1% digabung ke `OTHER` (CCLE) atau `Other` (GDSC); kategori `Unknown` tidak ikut digabung. Label asli `tissue_ccle` tetap disimpan. Variabel `hemato` (sel darah/limfoid) ditambahkan karena kelompok ini merupakan 17,0% sampel dan sering diperlakukan tersendiri. Variabel kategorikal tidak menjadi input klasterisasi sehingga tidak diperlukan pengkodean; bila nanti digunakan pada model prediktif, `OneHotEncoder(handle_unknown="ignore")` ditempatkan di dalam *pipeline* (lihat 1.7).

In [ ]:
def gabung_kategori_jarang(s, ambang_persen=1.0, label="OTHER", abaikan=("Unknown",)):
    persen = s.value_counts(normalize=True) * 100
    langka = persen[(persen < ambang_persen) & (~persen.index.isin(abaikan))].index
    return s.where(~s.isin(langka), label), list(langka)

info_sampel["tissue_major"], langka = gabung_kategori_jarang(info_sampel["tissue_ccle"])
info_sampel["hemato"] = info_sampel["tissue_ccle"].eq("HAEMATOPOIETIC_AND_LYMPHOID_TISSUE")
print("Digabung ke OTHER:", langka)
print("Jumlah kategori jaringan:", info_sampel["tissue_ccle"].nunique(), "->", info_sampel["tissue_major"].nunique())

for k in ["tissue", "msi", "growth", "medium"]:
    meta_bersih[k], _ = gabung_kategori_jarang(meta_bersih[k], label="Other")
print("\nKategori tissue GDSC setelah penggabungan:")
display(meta_bersih["tissue"].value_counts().to_frame("jumlah_sel").T)

## 1.5 Perbedaan Skala

**Masalah.** (a) Setelah log, varians antar gen masih sangat timpang (varians gen peringkat 1 = 15,96; peringkat 2.000 = 2,72; gen paling akhir 0,08). (b) `LN_IC50` memiliki skala berbeda antar obat: median per obat berkisar −5,39 sampai 10,42 (simpangan baku antar median 2,37), sedangkan median simpangan baku di dalam satu obat hanya 1,36.

**Mengapa penting.** Tanpa standardisasi, gen dengan varians besar mendominasi komponen utama. Pada respons obat, sel yang kebetulan diuji dengan obat berpotensi rendah tampak resisten semata-mata karena obatnya.

**Rencana penanganan.**
- **Ekspresi:** pilih 2.000 gen dengan varians `log2(TPM+1)` tertinggi, lalu standardisasi per gen (rata-rata 0, simpangan baku 1). Jumlah 2.000 dipilih karena pada uji awal akurasi 5-NN pada 20 PC relatif stabil untuk 1.000 sampai 5.000 gen (0,716 sampai 0,727), sedangkan penggunaan seluruh 19.809 gen menurunkannya menjadi 0,660. Selisih antar jumlah gen tersebut kecil dan berasal dari satu kali validasi silang, sehingga 1.000 dan 5.000 gen tetap diuji sebagai analisis sensitivitas.
- **Respons obat:** *z-score* `LN_IC50` per obat.

In [ ]:
# --- ekspresi: seleksi gen berbasis varians + standardisasi ---
X_log = expr_log.T
X_log.index.name = "kolom_ccle"
N_GEN   = 2000
var_gen = X_log.var(axis=0).sort_values(ascending=False)
X_top   = X_log[var_gen.index[:N_GEN]]
print(f"Varians gen peringkat 1: {var_gen.iloc[0]:.2f} | peringkat {N_GEN}: {var_gen.iloc[N_GEN-1]:.2f} | terakhir: {var_gen.iloc[-1]:.2f}")

X_std = pd.DataFrame(StandardScaler().fit_transform(X_top), index=X_top.index, columns=X_top.columns)
print("Setelah standardisasi -> rata-rata:", round(float(X_std.mean().mean()), 3), "| simpangan baku:", round(float(X_std.std().mean()), 3))

# --- respons: skala antar obat dan z-score per obat ---
median_obat = mat_resp.median()
print(f"\nMedian LN_IC50 per obat: {median_obat.min():.2f} sampai {median_obat.max():.2f} | std antar median: {median_obat.std():.2f}")
print(f"Median std di dalam obat: {mat_resp.std().median():.2f}")
z_resp = (mat_resp - mat_resp.mean()) / mat_resp.std()

## 1.6 Penyatuan GDSC dan CCLE

**Masalah.** Kedua dataset tidak memiliki kunci yang sama: GDSC memakai `COSMIC_ID` dan `CELL_LINE_NAME` (misalnya `PFSK-1`), sedangkan CCLE memakai nama kolom (misalnya `22RV1_PROSTATE`).

**Mengapa penting.** Pertanyaan 3 memerlukan data ekspresi dan respons dari sel yang sama; pasangan yang keliru akan merusak seluruh perbandingan.

**Rencana penanganan.** Nama sel dinormalisasi (huruf besar, hanya huruf dan angka), lalu digabungkan. Kunci yang ambigu (muncul lebih dari satu kali pada salah satu sisi) diselesaikan dengan kesesuaian jaringan. Pencocokan *fuzzy* tidak digunakan karena menghasilkan kandidat sel yang berbeda (misalnya `NCI-H64` dengan `NCIH647`).

In [ ]:
def kunci_nama(nama):
    return re.sub(r"[^A-Z0-9]", "", str(nama).upper())

ccle_map = info_sampel.reset_index()[["kolom_ccle", "nama_sel", "tissue_ccle"]].copy()
ccle_map["kunci"] = ccle_map["nama_sel"].map(kunci_nama)
gdsc_map = meta_bersih.reset_index()[["COSMIC_ID", "CELL_LINE_NAME", "tissue"]].copy()
gdsc_map["kunci"] = gdsc_map["CELL_LINE_NAME"].map(kunci_nama)

naif = gdsc_map.merge(ccle_map, on="kunci", how="inner")
kunci_ambigu = (set(ccle_map.loc[ccle_map["kunci"].duplicated(keep=False), "kunci"]) |
                set(gdsc_map.loc[gdsc_map["kunci"].duplicated(keep=False), "kunci"]))
print("Pencocokan naif:", len(naif), "baris | kunci ambigu:", len(kunci_ambigu))
display(naif[naif["kunci"].isin(kunci_ambigu)].sort_values("kunci")
        [["kunci", "COSMIC_ID", "CELL_LINE_NAME", "tissue", "kolom_ccle", "tissue_ccle"]])

kompatibel = {"thyroid": {"THYROID"},
              "aero_dig_tract": {"OESOPHAGUS", "UPPER_AERODIGESTIVE_TRACT", "SALIVARY_GLAND"},
              "lymphoma": {"HAEMATOPOIETIC_AND_LYMPHOID_TISSUE"}}

def lolos_jaringan(b):
    return True if b["kunci"] not in kunci_ambigu else b["tissue_ccle"] in kompatibel.get(b["tissue"], set())

cocok = naif[naif.apply(lolos_jaringan, axis=1)].copy()
assert cocok["COSMIC_ID"].is_unique and cocok["kolom_ccle"].is_unique
print(f"Sel GDSC: {len(gdsc_map)} | berhasil dicocokkan: {len(cocok)} ({len(cocok)/len(gdsc_map)*100:.1f}%)")

info_sampel = info_sampel.join(cocok.set_index("kolom_ccle")[["COSMIC_ID"]])
info_sampel["ada_respons"] = info_sampel["COSMIC_ID"].isin(mat_resp.index)
print("Sampel CCLE dengan data respons yang lolos ambang:", int(info_sampel["ada_respons"].sum()), "dari", len(info_sampel))

## 1.7 *Data Leakage*

***Data leakage*** terjadi ketika informasi dari data yang seharusnya tidak tersedia pada saat pelatihan (data uji, label, atau target) memengaruhi pembentukan fitur atau model, sehingga kinerja yang dilaporkan terlalu optimistis atau kesimpulan menjadi sirkular. Risiko pada proyek ini dan pencegahannya:

| Sumber potensi *leakage* | Contoh pada proyek | Risiko | Rencana pencegahan |
|---|---|---|---|
| Statistik praproses di-*fit* pada seluruh sampel | `StandardScaler`, seleksi 2.000 gen, PCA, *z-score* per obat, kuantil Q1/Q3 per obat | Informasi sampel uji ikut menentukan parameter transformasi | Untuk klasterisasi (tanpa label dan tanpa pemisahan latih-uji), *fit* pada seluruh sampel diterima dan dicatat sebagai keterbatasan. Untuk model prediktif (misalnya memprediksi sensitivitas obat), seluruh langkah di-*fit* hanya pada data latih melalui `Pipeline` |
| Label atau respons dipakai membentuk klaster | `tissue_ccle`, `LN_IC50` | Evaluasi menjadi sirkular: klaster tampak selaras dengan label karena label ikut membentuknya | Klaster hanya dibentuk dari `pc_*`. Label jaringan dan respons obat hanya digunakan setelah klaster terbentuk |
| Pemilihan hiperparameter dengan bantuan label | `N_GEN` dan `N_PC` dibandingkan melalui akurasi 5-NN terhadap `tissue_major` | Angka akurasi menjadi optimistis bila PCA di-*fit* pada seluruh sampel, termasuk lipatan validasi | Akurasi diperlakukan sebagai diagnostik, bukan ukuran kinerja akhir; pengaruhnya diperiksa pada sel kode di bawah |
| Variabel turunan dari target | `AUC` dan `Z_SCORE` berasal dari kurva dosis-respons yang sama dengan `LN_IC50` | Bila `LN_IC50` diprediksi, kedua variabel ini membawa informasi target | Tidak digunakan sebagai prediktor |
| Entitas yang sama muncul berulang | Satu *cell line* muncul pada banyak baris (sel × obat) | Pembagian acak per baris menempatkan sel yang sama pada data latih dan uji | Pembagian berkelompok (`GroupKFold`) berdasarkan `COSMIC_ID` |

Sel kode berikut membandingkan akurasi 5-NN pada 20 PC ketika (a) seleksi gen, standardisasi, dan PCA di-*fit* pada seluruh sampel, dan (b) ketiganya di-*fit* hanya pada data latih di setiap lipatan. Selisih keduanya menunjukkan besarnya pengaruh *leakage* pada diagnostik tersebut.

In [ ]:
class PilihGenVarians(BaseEstimator, TransformerMixin):
    """Memilih n_gen gen dengan varians terbesar; dihitung hanya dari data yang diberikan ke fit()."""
    def __init__(self, n_gen=2000):
        self.n_gen = n_gen
    def fit(self, X, y=None):
        self.idx_ = np.argsort(np.asarray(X).var(axis=0))[::-1][: self.n_gen]
        return self
    def transform(self, X):
        return np.asarray(X)[:, self.idx_]

label = info_sampel["tissue_major"].reindex(X_log.index)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# (a) praproses di-fit pada SELURUH sampel (dapat bocor ke lipatan validasi)
pcs20_semua = PCA(n_components=20, random_state=42).fit_transform(X_std)
acc_a = cross_val_score(KNeighborsClassifier(5), pcs20_semua, label, cv=cv).mean()

# (b) praproses di-fit hanya pada data latih di setiap lipatan
pipa = Pipeline([("gen", PilihGenVarians(N_GEN)), ("std", StandardScaler()),
                 ("pca", PCA(n_components=20, random_state=42)), ("knn", KNeighborsClassifier(5))])
acc_b = cross_val_score(pipa, X_log.values, label, cv=cv).mean()

print(f"Akurasi 5-NN, 20 PC (a) praproses pada seluruh sampel : {acc_a:.3f}")
print(f"Akurasi 5-NN, 20 PC (b) praproses di dalam lipatan   : {acc_b:.3f}")
print(f"Selisih (a) - (b)                                    : {acc_a - acc_b:+.3f}")

# BAGIAN 2. *INITIAL FEATURE ENGINEERING*

*Initial feature engineering* bertujuan menyusun fitur awal yang relevan dengan masalah proyek dan dapat diverifikasi, bukan fitur final. Seluruh fitur dibentuk dari data yang sudah dibersihkan pada Bagian 1.

## 2.0 Kandidat Fitur

| Kandidat Fitur | Dibentuk dari | Jenis | Alasan |
|---|---|---|---|
| `pc_1` sampai `pc_20` | 2.000 gen dengan varians `log2(TPM+1)` tertinggi, distandardisasi per gen, lalu direduksi dengan PCA | Transformasi (reduksi dimensi) | Meringkas ribuan gen yang saling berkorelasi menjadi sedikit dimensi sebagai input klasterisasi dan menjawab pertanyaan 1 dan 2. Lingkup 20 PC dipilih berdasarkan diagnostik pada 2.1 |
| `resp_mean_z` | `LN_IC50` distandardisasi per obat, kemudian dirata-ratakan per *cell line* (nilai kosong diabaikan) | Transformasi dan agregasi | Menyatakan kecenderungan resistensi sel secara terkoreksi terhadap skala obat sehingga sel yang diuji dengan panel obat berbeda tetap sebanding. Digunakan untuk pertanyaan 3 |
| `resp_frac_sensitif` | Jumlah obat dengan `LN_IC50` di bawah kuartil pertama obat tersebut dibagi jumlah obat yang diuji pada sel itu | Rasio | Berupa proporsi sehingga tidak bergantung pada banyaknya obat yang diuji dan lebih mudah ditafsirkan daripada rata-rata. Digunakan untuk pertanyaan 3 |
| `tissue_major` | `tissue_ccle` dengan kategori < 1% digabung ke `OTHER` (dibentuk pada 1.4) | Pengelompokan nilai | Menyediakan label evaluasi yang stabil untuk menilai kesesuaian klaster dengan jaringan. Bukan input klasterisasi |

`pc_*` adalah satu-satunya input klasterisasi. `resp_mean_z`, `resp_frac_sensitif`, dan `tissue_major` digunakan setelah klaster terbentuk.

## 2.1 Fitur Ekspresi: `pc_1` sampai `pc_20`

Jumlah PC ditentukan dengan dua ukuran: varians kumulatif dan akurasi 5-NN terhadap `tissue_major` (validasi silang 5-lipat). Varians kumulatif saja tidak memadai karena varians dapat tinggi akibat *noise*; akurasi 5-NN hanya berperan sebagai diagnostik, bukan pembentuk klaster (lihat catatan *leakage* pada 1.7).

In [ ]:
pca = PCA(n_components=50, random_state=42)
pcs = pca.fit_transform(X_std)
var_rasio = pca.explained_variance_ratio_
var_kum   = var_rasio.cumsum()
print("Varians PC1-PC5 (%):", (var_rasio[:5] * 100).round(2))

baris = []
for k in [2, 5, 10, 20, 30, 50]:
    acc = cross_val_score(KNeighborsClassifier(5), pcs[:, :k], label, cv=cv).mean()
    baris.append({"n_pc": k, "var_kumulatif_%": round(var_kum[k-1] * 100, 1), "akurasi_5NN": round(acc, 3)})
display(pd.DataFrame(baris).set_index("n_pc"))

N_PC = 20
fitur_A = pd.DataFrame(pcs[:, :N_PC], index=X_std.index, columns=[f"pc_{i}" for i in range(1, N_PC + 1)])
print(f"Fitur ekspresi: {fitur_A.shape} | varians kumulatif {N_PC} PC: {var_kum[N_PC-1]*100:.1f}%")

**Temuan.** PC1 menjelaskan 17,4% dan PC2 12,7% varians (total 30,1%); 50 PC menjelaskan 67,6%. Akurasi 5-NN meningkat tajam hingga sekitar 20 PC (0,449 pada 2 PC; 0,672 pada 10 PC; 0,721 pada 20 PC) lalu melandai (0,735 pada 30 PC; 0,750 pada 50 PC). Informasi jaringan dengan demikian tersebar pada banyak komponen, bukan hanya dua PC pertama. Jawaban awal bagi pertanyaan 1 adalah sekitar 10 sampai 20 dimensi; `N_PC = 20` dipakai sebagai titik awal dan dikonfirmasi ulang setelah klasterisasi. Angka akurasi tersebut berasal dari praproses yang di-*fit* pada seluruh sampel, sehingga perlu dibaca bersama hasil pada 1.7.

## 2.2 Fitur Respons Obat: `resp_mean_z` dan `resp_frac_sensitif`

Kedua fitur dihitung dari `mat_resp` dan `z_resp` (Bagian 1). Fitur ini tidak boleh menjadi input klasterisasi, karena perbandingan respons antarklaster akan menjadi sirkular.

In [ ]:
n_uji = mat_resp.notna().sum(axis=1)
q1_obat = mat_resp.quantile(0.25)

fitur_B = pd.DataFrame({
    "resp_n_obat":        n_uji,                                                   # penyebut / jumlah obat teruji
    "resp_mean_z":        z_resp.mean(axis=1),                                     # transformasi + agregasi
    "resp_frac_sensitif": mat_resp.lt(q1_obat, axis=1).sum(axis=1) / n_uji,        # rasio
})
display(fitur_B.describe().round(3))

# bukti manfaat koreksi per obat: bandingkan dengan rata-rata mentah
banding = pd.DataFrame({"mean_mentah": mat_resp.mean(axis=1), "mean_z": fitur_B["resp_mean_z"], "n_obat": n_uji})
banding["geser_peringkat"] = (banding["mean_mentah"].rank() - banding["mean_z"].rank()).abs()
kecil = banding["n_obat"] < 200
print("Spearman rata-rata mentah vs resp_mean_z:", round(banding["mean_mentah"].corr(banding["mean_z"], method="spearman"), 3))
print("Median pergeseran peringkat | n_obat < 200:", banding.loc[kecil, "geser_peringkat"].median(),
      "| n_obat >= 200:", banding.loc[~kecil, "geser_peringkat"].median())
print("Sel bergeser > 50 peringkat:", int((banding["geser_peringkat"] > 50).sum()), "dari", len(banding))
print("Spearman resp_mean_z vs resp_frac_sensitif:",
      round(fitur_B["resp_mean_z"].corr(fitur_B["resp_frac_sensitif"], method="spearman"), 3))

**Temuan.** Jumlah obat yang diuji per sel bervariasi (minimum 145, median 269, maksimum 270). Korelasi Spearman antara rata-rata mentah dan `resp_mean_z` sangat tinggi (0,993), tetapi 147 dari 965 sel bergeser lebih dari 50 peringkat; median pergeseran 40 pada 249 sel dengan `n_obat` < 200 dibandingkan 17 pada sel lainnya. Dengan demikian koreksi per obat terutama penting bagi sel dengan panel obat kecil, sehingga `resp_mean_z` lebih layak digunakan daripada rata-rata mentah. Hubungan `resp_frac_sensitif` dengan `resp_mean_z` pada sel kode di atas menunjukkan apakah kedua fitur redundan; bila korelasinya sangat tinggi, salah satunya cukup dipertahankan.

## 2.3 Tabel Fitur Akhir

Dua tabel disusun sesuai kebutuhan pertanyaan penelitian: **`fitur_semua`** (seluruh sampel CCLE; pertanyaan 1, 2, 4) dan **`fitur_gabungan`** (sampel yang memiliki data respons; pertanyaan 3). Awalan kolom (`pc_`, `resp_`) menunjukkan asal fitur.

In [ ]:
anotasi = info_sampel[["tissue_ccle", "tissue_major", "hemato"]]

peta_id = cocok.set_index("COSMIC_ID")["kolom_ccle"]
fitur_B_g = fitur_B.loc[fitur_B.index.intersection(peta_id.index)].copy()
fitur_B_g.index = peta_id.loc[fitur_B_g.index].values
fitur_B_g.index.name = "kolom_ccle"

fitur_semua    = fitur_A.join(anotasi)
fitur_gabungan = fitur_A.join(fitur_B_g, how="inner").join(anotasi)

num = fitur_gabungan.filter(regex=r"^(pc_|resp_)")
print("fitur_semua    :", fitur_semua.shape)
print("fitur_gabungan :", fitur_gabungan.shape)
print("NaN pada fitur numerik (gabungan):", int(num.isna().sum().sum()), "| Inf:", int(np.isinf(num).sum().sum()))

## 2.4 Ringkasan dan Langkah Berikutnya

**Keluaran.** `fitur_semua` dan `fitur_gabungan`, serta `expr_log`, `X_std`, dan `z_resp` sebagai bahan analisis lanjutan.

**Langkah berikutnya.**
1. Menetapkan jumlah PC final dan menguji kepekaan terhadap `N_GEN` (1.000, 2.000, 5.000).
2. Klasterisasi hanya pada `pc_*` (K-Means, hierarchical, atau GMM); jumlah klaster dipilih dengan *silhouette* dan *elbow*; kesesuaian dengan jaringan dinilai dengan tabel silang dan *adjusted Rand index*.
3. Membandingkan `resp_mean_z` dan `resp_frac_sensitif` antarklaster, juga di dalam kelompok jaringan agar tidak sekadar mengulang perbedaan jaringan.
4. Analisis ekspresi diferensial antarklaster dan pemetaan kode Ensembl ke simbol gen.

**Keterbatasan.**
- Korelasi antara klaster dan respons obat tidak menunjukkan hubungan sebab-akibat.
- *Cell line* bukan pasien, dan kadar mRNA tidak selalu mencerminkan kadar protein.
- Jumlah sampel terbatas, terutama pada subset hematopoietik.
- Informasi *batch* tidak tersedia.
- Ambang awal (10% sampel, 2.000 gen, 20 PC, batas kosong 50%) merupakan pilihan awal yang perlu diuji kepekaannya.
- Pencocokan sel GDSC dan CCLE berbasis nama; verifikasi dengan identitas resmi (misalnya Cellosaurus atau DepMap) dianjurkan.